# 00 · Preprocessing — profiling and cleaning the raw sources

**Task.** Business Entity Resolution: for every Source-1 (S1) business, find all records in
Source 2 (S2) and Source 3 (S3) describing the same real-world business. S1 is deduplicated; an
S1 entity can match zero, one or many S2/S3 records. Leaderboard metric: **F0.5 per S1 entity,
macro-averaged** (singletons included — an empty prediction for an entity with no match scores 1.0).

**Raw schema** (tab-separated, no quoting): `entity_id`, `business_name`, `business_address`,
`country`. Training covers US + India; **the test set adds France**, so nothing may hard-code
country values.

This notebook profiles each raw source, explains the cleaning rules (implemented once in
`src/ber/cleaning.py`), and streams every source through them into Parquet. Notebook 05 calls
the **same** functions on the test split, guaranteeing identical preprocessing at inference.

In [ ]:
import sys, time, gc
from pathlib import Path

# notebooks/ -> ../src holds the shared, importable pipeline package `ber`
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ber import config as C, io, pipeline as P

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
C.ensure_dirs()
print(C.describe())
try:
    import psutil
    print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB | CPUs: {psutil.cpu_count()}")
except ImportError:
    pass

> **Infrastructure & cost.** Built to run the *full* dataset on a free-tier-class EC2 instance
> (m7i-flex.large: 2 vCPU, 8 GB RAM + 4 GB swap, 30 GB gp3) — every step streams or chunks
> its data. Checkpoints go to the local EBS volume (which survives stop/start) and small,
> valuable artifacts (model, metadata, submission) are mirrored to S3; big intermediates are
> kept local to stay inside the S3 free tier (5 GB, ~2k PUTs/month). S3→EC2 transfer in the
> same region is free. Re-running a notebook skips stages whose checkpoint exists
> (set `FORCE = True` to recompute). Everything is CPU-bound; no GPU is used.

## 1. Profile the raw sources

Each source is loaded (from the local cache; downloaded from S3 on first use), profiled, and
released before the next one to keep memory flat.

| check | why it matters |
|---|---|
| empty names / addresses | whether a record can be matched on one field only |
| duplicate ids / rows | ids must be unique for the submission; duplicate rows inflate candidates |
| name length & tokens | drives n-gram choice, flags truncation |
| non-Latin script share | Devanagari / Kannada / Telugu names need transliteration |
| `fka/dba`, web domains, `(ID: …)`, leading junk, `NULL` | each needs a dedicated cleaning rule |

In [ ]:
FORCE = False
SPLIT = "train"
NON_LATIN = "[^\u0000-\u024F\u1E00-\u1EFF]"

def profile(df: pd.DataFrame, name: str) -> dict:
    n, a = df["business_name"].astype(str), df["business_address"].astype(str)
    return {
        "source": name, "rows": len(df),
        "dup_entity_id": int(df["entity_id"].duplicated().sum()),
        "dup_full_rows": int(df.duplicated(["business_name", "business_address", "country"]).sum()),
        "empty_name_%": round(100 * (n.str.strip() == "").mean(), 3),
        "empty_addr_%": round(100 * (a.str.strip() == "").mean(), 3),
        "name_len_p50": n.str.len().median(), "name_len_p99": n.str.len().quantile(0.99),
        "name_tok_p50": n.str.split().str.len().median(), "addr_len_p50": a.str.len().median(),
        "non_latin_name_%": round(100 * n.str.contains(NON_LATIN).mean(), 2),
        "non_latin_addr_%": round(100 * a.str.contains(NON_LATIN).mean(), 2),
        "leading_junk_%": round(100 * n.str.match(r"^[^\w]").mean(), 2),
        "html_entity_%": round(100 * n.str.contains("&amp;|&#").mean(), 3),
        "alias_marker_%": round(100 * n.str.lower().str.contains(r"\b(?:fka|f/k/a|dba|d/b/a|aka)\b").mean(), 2),
        "web_domain_%": round(100 * n.str.lower().str.contains(r"\.(?:com|in|net|org|fr)\b").mean(), 2),
        "id_tag_%": round(100 * n.str.contains(r"\(ID:", case=False).mean(), 2),
        "addr_NULL_%": round(100 * a.str.contains(r"\bNULL\b").mean(), 2),
        "countries": df["country"].value_counts().to_dict(),
    }

profiles, samples = [], []
for k in ("s1", "s2", "s3"):
    df = io.read_raw(SPLIT, k, usecols=P.SRC_COLS)
    profiles.append(profile(df, k))
    samples.append(df.sample(8, random_state=1))
    if k == "s1":
        print("dtypes:", df.dtypes.to_dict())
        df["business_name"].str.len().clip(upper=80).hist(bins=80, figsize=(8, 3)); plt.title("S1 raw name length"); plt.show()
    del df; gc.collect()
pd.DataFrame(profiles).set_index("source").T

### Noise examples (random S1 / S2 / S3 rows)

In [ ]:
pd.concat(samples)[P.SRC_COLS]

## 2. Cleaning design (`src/ber/cleaning.py`)

**Names**
- Transliterate to ASCII with `anyascii` (ISC licence): `श्री प्रोजेक्ट्स प्राइवेट लिमिटेड → sri projekts praivet limited`,
  `Góld → gold`, so Indic-script and accented records stay matchable against Latin spellings.
- Lower-case; `&`→`and`; strip punctuation; collapse initials (`L.L.C.`→`llc`, `P.C.`→`pc`).
- Remove tracking noise: trailing `| www.site.com`, `(ID: 78977)`; bare web domains
  (`victorylaboratories.com`) become the domain stem and are flagged `name_is_domain`.
- Fix digit-for-letter typos inside words (`N0rth`→`north`, `5ons`→`sons`); dedupe repeated tokens.
- Canonicalise legal forms (`Limited/Ltd`→`ltd`, `Private/Pvt`→`pvt`, `SARL`, `SAS`, …).
- **Alias markers** (`fka`, `dba`, `aka`, `formerly`): `name_core` = the part after the marker,
  `name_alt` = the part before; features compare both.

**Stopword decision.** Only legal forms and honorifics (`mr`, `smt`, `the`, …) are removed, and
only from `name_core`. Generic business words (`services`, `center`, `clinic`) are **kept**: they
discriminate `Grace Chapel` from `Grace Clinic`; the noise that appends them
(`Bright Barbershop Ventures`) is absorbed by token-set / partial similarity features. The full
normalised name (`name_clean`, legal forms kept) is retained too.

**Addresses**
- Drop non-Latin runs *before* transliteration (native-script state names like `महाराष्ट्र`).
- A whole comma-separated component that is a state/province (`TX`, `Texas`, `MH`, `Maharashtra`)
  moves to `state` (codes resolved with the record's own country; unknown countries keep the raw
  token) and leaves `addr_clean`.
- Strip leading zeros (`00272`→`272`), ordinals (`3rd`→`3`), PO Box / PMB numbers and formatting
  tokens (`Door No`, `#`, `H.No`, `NULL`, `Unit`, `Floor`); expand abbreviations (`St`→`street`,
  `Rd`→`road`, `R.`→`rue`, `Bombay`→`mumbai` …).
- `addr_nums`: numeric tokens (house / plot / flat numbers); `postcode`: a 5–6 digit number that
  ends the address (a leading 5-digit number is usually a US house number).

**Nulls / empties.** No row is dropped: every S1 entity must appear in the submission and S2/S3
records with an empty address (~3–4%) are still matchable by name. Empty fields stay `""`, are
exposed as explicit flags, and their similarity features are NaN (LightGBM handles NaN natively).

In [ ]:
from ber.cleaning import clean_name, clean_address
rows = []
for _, r in pd.concat(samples).iterrows():
    n, a = clean_name(r.business_name), clean_address(r.business_address, r.country.lower())
    rows.append({"raw_name": r.business_name, "name_core": n["name_core"], "name_alt": n["name_alt"],
                 "legal": n["legal"], "raw_address": r.business_address, "addr_clean": a["addr_clean"],
                 "nums": a["addr_nums"], "state": a["state"]})
pd.DataFrame(rows)

## 3. Clean all sources (streamed) and checkpoint

Raw TSVs are read in chunks of `C.CLEAN_CHUNK_ROWS`, cleaned in `C.N_JOBS` worker processes and
appended to `clean/{split}_{s1,s2,s3}.parquet`; the ground truth is saved as
`clean/train_gt.parquet`. Peak memory stays well under 1 GB.

In [ ]:
t0 = time.time()
P.run_clean(SPLIT, force=FORCE)
print(f"done in {time.time() - t0:.0f}s")

## 4. Post-cleaning sanity checks

In [ ]:
def post_stats(k):
    df = pd.read_parquet(io.fetch(f"clean/{SPLIT}_{k}.parquet"),
                         columns=["name_core", "addr_clean", "name_alt", "name_nonlatin", "name_is_domain",
                                  "state", "postcode", "addr_nums", "name_ntok", "addr_ntok"])
    out = {"source": k, "rows": len(df),
           "empty_name_core_%": round(100 * (df.name_core == "").mean(), 3),
           "empty_addr_clean_%": round(100 * (df.addr_clean == "").mean(), 3),
           "has_alias_%": round(100 * (df.name_alt != "").mean(), 3),
           "nonlatin_name_%": round(100 * df.name_nonlatin.mean(), 3),
           "domain_name_%": round(100 * df.name_is_domain.mean(), 3),
           "has_state_%": round(100 * (df.state != "").mean(), 2),
           "has_postcode_%": round(100 * (df.postcode != "").mean(), 2),
           "has_numbers_%": round(100 * (df.addr_nums != "").mean(), 2),
           "name_ntok_p50": df.name_ntok.median(), "addr_ntok_p50": df.addr_ntok.median()}
    del df; gc.collect()
    return out

pd.DataFrame([post_stats(k) for k in ("s1", "s2", "s3")]).set_index("source").T

In [ ]:
pd.read_parquet(io.fetch(f"clean/{SPLIT}_s2.parquet")).head(10)